# Analiza Forex - Midas Project

Advanced ML Analysis with XGBoost, LSTM, and Nearest Neighbors.

In [1]:
import plotly.graph_objects as go
import pandas as pd
import config
from data_connector import MT5Connector
from database import DatabaseManager
from analysis import Analyzer

# Global Signal Initialization to prevent NameError
climax_signals = effort_signals = forex_scenarios = pd.DataFrame()
multi_extremes = sot_signals = hinge_signals = storyboard_signals = pd.DataFrame()
springboard_signals = pd.DataFrame()
prediction = None

# Settings
pd.set_option('display.max_columns', None)

C:\Users\donniebrasco\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Data Connection

In [2]:
connector = MT5Connector()
db = DatabaseManager()
symbol = 'XAUUSD'
corr_symbol = 'XAUUSD' if symbol == 'XAGUSD' else 'XAGUSD'

df = db.load_candles(symbol, limit=50000)
df_corr = db.load_candles(corr_symbol, limit=50000)

if df.empty:
    print("No data. Run main.py first.")
else:
    print(f"Loaded {len(df)} candles for {symbol}.")
    if not df_corr.empty:
        print(f"Loaded {len(df_corr)} correlated candles for {corr_symbol}.")

Loaded 9099 candles for XAUUSD.
Loaded 9098 correlated candles for XAGUSD.


## 2. Analysis (ZigZag & Swings)

In [3]:
if not df.empty:
    df['zigzag'] = Analyzer.calculate_zigzag(df)
    swings = Analyzer.analyze_swings(df)
    
    # Forex Statistical Metrics
    df = Analyzer.calculate_forex_stats(df)
    db.save_forex_metrics(df, symbol)
    
    print(f"Swings found: {len(swings)}")
    print("Forex statistical metrics calculated and saved.")

Swings found: 546
Forex statistical metrics calculated and saved.


C:\Users\donniebrasco\Documents\PROJEKT\database.py:178: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metrics_df['time'] = pd.to_datetime(metrics_df['time'])


## 2b. SMT Divergence Testing
Analyze correlations and divergences between related assets.

In [ ]:
if not df.empty and not df_corr.empty:
    print(f"Visualizing SMT Divergence: {symbol} vs {corr_symbol}...")
    fig_smt = Analyzer.plot_smt_divergence(df, df_corr, symbol1=symbol, symbol2=corr_symbol)
    if fig_smt:
        fig_smt.show()
    else:
        print("Not enough overlapping data for SMT visualization.")

## 3. Advanced ML Predictions

In [ ]:
# Initialize variables to prevent NameError
climax_signals = pd.DataFrame()
effort_signals = pd.DataFrame()
forex_scenarios = pd.DataFrame()
multi_extremes = pd.DataFrame()
sot_signals = pd.DataFrame()
hinge_signals = pd.DataFrame()
springboard_signals = pd.DataFrame()

# Model Selection for Live Prediction
MODEL_TYPE = 'XGBoost'  # Options: 'NN', 'XGBoost', 'LSTM'
prediction = None
if not swings.empty:
    print(f"Making live prediction using {MODEL_TYPE}...")
    if MODEL_TYPE == 'NN':
        prediction = Analyzer.predict_next_swing_nn(swings, k=5)
    elif MODEL_TYPE == 'XGBoost':
        prediction = Analyzer.predict_next_swing_xgboost(swings, window=75)
    elif MODEL_TYPE == 'LSTM':
        prediction = Analyzer.predict_next_swing_lstm(swings, window=6)

    if prediction:
        print(f"\n=== {MODEL_TYPE} PROGNOZA ===")
        print(f"Kierunek: {prediction['direction']}")
        print(f"Cel cenowy: {prediction['target_price']:.2f}")
        print(f"Przewidywany czas: {prediction['target_time']}")

# Behavioral & Exhaustion Analysis
print("Detecting anomalies and exhaustion (SMT integrated)...")
climax_signals = Analyzer.detect_climax(df)
effort_signals = Analyzer.analyze_effort_vs_result(swings)
forex_scenarios = Analyzer.detect_forex_scenarios(df)
multi_extremes = Analyzer.detect_multi_extremes(swings)

# Calculate Exhaustion Score using Correlated Data for SMT Divergence
df = Analyzer.calculate_exhaustion_score(df, swings, df_corr=df_corr)

# Save behavioral signals to DB
if not climax_signals.empty: db.save_behavioral_signals(climax_signals, symbol)
if not effort_signals.empty: db.save_behavioral_signals(effort_signals, symbol)
if not forex_scenarios.empty: db.save_behavioral_signals(forex_scenarios, symbol)
if not multi_extremes.empty: db.save_behavioral_signals(multi_extremes, symbol)

# Wave Logic Signals
sot_signals = Analyzer.detect_sot(swings)
hinge_signals = Analyzer.detect_hinge(swings)
springboard_signals = Analyzer.detect_springboard(swings, hinge_signals) if not hinge_signals.empty else pd.DataFrame()

## 4. Model Backtesting & Verification

In [ ]:
BT_MODEL = 'XGBoost' # Try 'NN', 'XGBoost', or 'LSTM'

backtest_results = pd.DataFrame()
if not swings.empty:
    print(f"Backtesting {BT_MODEL} (this may take a few seconds)...")
    backtest_results = Analyzer.backtest_walk_forward(swings, method=BT_MODEL, window=10, train_size=200, test_size=50)
    
    if not backtest_results.empty:
        scores = Analyzer.verify_performance(backtest_results)
        print(f"\n=== {BT_MODEL} SCORES ===")
        print(f"Direction Accuracy: {scores['direction_accuracy']*100:.1f}%")
        print(f"Price MAPE: {scores['price_mape']*100:.2f}%")
        if "sharpe_ratio" in scores:
            print(f"Sharpe Ratio: {scores['sharpe_ratio']:.2f}")
            print(f"Max Drawdown: {scores['max_drawdown']*100:.2f}%")
            print(f"Win Rate: {scores['win_rate']*100:.1f}%")
        print(f"Range MAPE: {scores['range_mape']*100:.2f}%")
        print(f"Duration MAPE: {scores['duration_mape']*100:.2f}%")
        
        # Save and display report
        csv_rep, md_rep = Analyzer.save_backtest_report(backtest_results, scores, BT_MODEL, db_manager=db, symbol=symbol)
        print(f"\nReport saved to: {csv_rep}")
        print(f"Summary saved to: {md_rep}")
        
        display(backtest_results.tail(3))
    else:
        print("No backtest results.")


## 5. Visualization

In [4]:
from plotly.subplots import make_subplots
if not df.empty:
    df_v = df.iloc[max(0, len(df)-600):]
    
    # Create Subplots: Main Chart + Exhaustion Panel
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, row_heights=[0.7, 0.3])
    
    # Candle Chart
    fig.add_trace(go.Candlestick(x=df_v['time'], open=df_v['open'], high=df_v['high'], low=df_v['low'], close=df_v['close'], name='Price', opacity=0.4), row=1, col=1)
    
    # ZigZag
    zz_pts = df_v[df_v['zigzag'] != 0]
    fig.add_trace(go.Scatter(x=zz_pts['time'], y=zz_pts['zigzag'], mode='lines+markers', line=dict(color='blue', width=2), name='ZigZag'), row=1, col=1)

    # Exhaustion Score Panel
    fig.add_trace(go.Scatter(x=df_v['time'], y=df_v.get('exhaustion_score', pd.Series(0, index=df_v.index)), fill='tozeroy', name='Exhaustion Score', line=dict(color='red', width=2)), row=2, col=1)
    fig.add_hline(y=70, line_dash="dash", line_color="orange", row=2, col=1)
    fig.update_yaxes(title_text="Exhaustion %", range=[0, 100], row=2, col=1)

    # Behavioral Signals Overlays
    behavioral_cfg = [
        (locals().get('climax_signals', pd.DataFrame()), 'white', 'hexagram', 'Climax'),
        (locals().get('effort_signals', pd.DataFrame()), 'orange', 'diamond-tall', 'Effort-vs-Result'),
        (locals().get('forex_scenarios', pd.DataFrame()), 'coral', 'star-triangle-up', 'Wyckoff Test'),
        (locals().get('multi_extremes', pd.DataFrame()), 'yellow', 'triangle-up-dot', 'Multi-Extreme')
    ]
    for sigs, color, marker, name in behavioral_cfg:
        if not sigs.empty:
            vis = sigs[sigs['time'] >= df_v['time'].iloc[0]]
            if not vis.empty:
                fig.add_trace(go.Scatter(x=vis['time'], y=vis['price'], mode='markers', 
                                         marker=dict(color=color, size=12, symbol=marker), name=name), row=1, col=1)

    # Wave Logic
    for sigs, color, marker, name in [(locals().get('sot_signals', pd.DataFrame()), 'red', 'triangle-down', 'SOT'), (locals().get('hinge_signals', pd.DataFrame()), 'white', 'diamond', 'Hinge'), (locals().get('springboard_signals', pd.DataFrame()), 'gold', 'star', 'Springboard')]:
        if not sigs.empty:
            vis = sigs[sigs['time'] >= df_v['time'].iloc[0]]
            if not vis.empty:
                fig.add_trace(go.Scatter(x=vis['time'], y=vis['price'], mode='markers', marker=dict(color=color, size=14, symbol=marker), name=name), row=1, col=1)

    fig.update_layout(title=f'{symbol} Exhaustion & Behavioral Analysis', template='plotly_dark', 
                      xaxis_rangeslider_visible=False, height=900, showlegend=True)
    fig.show()


NameError: name 'climax_signals' is not defined

## 6. Model Performance Comparison

Compare results from different model runs saved in the database.

In [ ]:
perf_df = db.load_performance_comparison()
if not perf_df.empty:
    display(perf_df.sort_values('timestamp', ascending=False).head(10))
    
    # Visual comparison
    import plotly.express as px
    fig_cmp = px.bar(perf_df, x='model_name', y='direction_accuracy', color='model_name', 
                     title='Model Direction Accuracy Comparison', barmode='group')
    fig_cmp.update_layout(template='plotly_dark')
    fig_cmp.show()
else:
    print("No performance metrics found in database.")

## 7. Analytics Dashboard
Quick overview of market parameters and anomaly heatmap.

In [ ]:
import numpy as np
print("=== MIDAS ANALYTICS DASHBOARD ===")
if not df.empty:
    last_row = df.iloc[-1]
    print(f"Current Symbol: {symbol}")
    print(f"Exhaustion Score: {last_row.get('exhaustion_score', 0):.1f}%")
    print(f"Daily Z-Score: {last_row.get('daily_zscore', 0):.2f}")
    print(f"Vol Trend (30): {last_row.get('vol_trend30', 0):.2f}")
    
    # Signal Heatmap (Recent 50 bars)
    recent_signals = []
    if not climax_signals.empty: recent_signals.append(('Climax', len(climax_signals)))
    if not effort_signals.empty: recent_signals.append(('Effort', len(effort_signals)))
    if not multi_extremes.empty: recent_signals.append(('Multi-Ext', len(multi_extremes)))
    
    print("\nRecent Behavioral Events:")
    for name, count in recent_signals:
        print(f"- {name}: {count} detected")

## 8. Advanced Production Analysis (V2)

This section uses `analysis_v2.py` which includes:
- **VSA/Wyckoff Features**: Advanced volume and spread analysis.
- **Enriched Swings**: Swing-level data enriched with bar-level statistics.
- **Verified No-Leakage**: Strict temporal separation for ML training.

In [ ]:
from analysis_v2 import Analyzer as AnalyzerV2, ForexFeatures, ValidationTools
import pandas as pd

# 1. Calculate Comprehensive Forex Features
print("Calculating V2 features...")
df_v2 = ForexFeatures.calculate_all(df)

# 2. Analyze Swings (ZigZag)
df_v2['zigzag'] = AnalyzerV2.calculate_zigzag(df_v2)
swings_v2 = AnalyzerV2.analyze_swings(df_v2)

# 3. Enrich Swings with Forex Features (Critical Tier)
swings_enriched = AnalyzerV2.enrich_swings_with_forex_features(swings_v2, df_v2, feature_tier='critical')

#print(f"\nEnriched Swings: {len(swings_enriched)}")
display(swings_enriched.tail())

In [ ]:
# 4. Backtest V2 Logic (XGBoost)
BT_MODEL_V2 = 'XGBoost'
print(f"Running V2 Backtest ({BT_MODEL_V2})...")
results_v2 = AnalyzerV2.backtest(swings_enriched, df_v2, method=BT_MODEL_V2, window=10, min_history=50)

if not results_v2.empty:
    scores_v2 = AnalyzerV2.verify_performance(results_v2)
    print(f"\n=== {BT_MODEL_V2} V2 SCORES ===")
    for k, v in scores_v2.items():
        val = v*100 if 'mape' in k or 'accuracy' in k else v
        suffix = '%' if 'mape' in k or 'accuracy' in k else ''
        print(f"{k.replace('_', ' ').title()}: {val:.2f}{suffix}")
    
    # Save Report
    AnalyzerV2.save_backtest_report(results_v2, scores_v2, f"{BT_MODEL_V2}_V2", symbol=symbol)
else:
    print("Backtest failed or insufficient data.")

In [ ]:
# 5. Data Leakage Verification
print("Running Leakage Verification..`.")
ValidationTools.test_data_leakage(swings_enriched, df_v2, window=10)

# 9. Reversal Analytics Dashboard (V3)
---
This compartment visualizes high-probability reversal zones by correlating price multi-extremes with VSA/Wyckoff signatures.

In [ ]:
from analysis_v2 import Analyzer, ForexFeatures, DashboardBuilder
import pandas as pd
from IPython.display import display

# 1. Detect Reversals using the new V3 logic
print("Detecting reversal signatures (Price + VSA)...")
reversals_df = Analyzer.detect_reversal_signatures(swings_v2, df_v2, tolerance=0.015)

# 2. Display the Reversal Dashboard
# Note: USES df_v2 and swings_v2 from Section 8
SYMBOL_DS = SYMBOL if "SYMBOL" in globals() else "Unknown"
fig_reversal = DashboardBuilder.plot_reversal_analytics(df_v2, reversals_df, symbol=SYMBOL_DS)
fig_reversal.show()

# 3. Full Spectrum Analytics Table
if not reversals_df.empty:
    print("\nFull Spectrum Reversal Table:")
    # Highlight by strength and climax score
    display(reversals_df.sort_values("strength", ascending=False).style.background_gradient(subset=["strength", "climax_score"], cmap="RdYlGn"))
else:
    print("No reversal patterns detected in the current window.")

## 9. Swing Analytics Dashboard (V2.0)

Comprehensive swing structure analysis, pullback depth, and volume distribution for Gold (XAUUSD).

In [ ]:
from swing_analytics import SwingAnalytics, SWING_TYPE, TREND_STATE
from swing_dashboard import SwingDashboard
import pandas as pd

if not df.empty:
    # 1. Analytics Setup
    point = 0.01 if symbol == 'XAUUSD' else 0.00001
    analytics = SwingAnalytics(point_value=point)
    
    # 2. Process Data
    zigzag = analytics.calculate_zigzag(df)
    swings_v2 = analytics.process_swings(df, zigzag)
    pullbacks_v2 = analytics.analyze_pullbacks(swings_v2)
    
    # 3. Interactive Plotly chart with Predictions
    pred = locals().get('prediction')
    fig_swings = analytics.plot_swings(df.iloc[-500:], swings_v2[-25:], prediction=pred, title=f'Interactive Swing Map - {symbol}')
    fig_swings.show()
    
    # 4. Rich Analytical Tables
    dashboard = SwingDashboard(symbol, 'M1')
    dashboard.render_all_tables(swings_v2, pullbacks_v2)